![](https://i.ibb.co/7t6wKN7/000.png)

<div class="alert alert-block alert-info" style="font-size:20px; background-color: #f4ffab; font-family:verdana; color: #8a0f0f; border: 3px #cbd600 solid">
    <b>Hello friends 🙃</b>
    <br>There is nothing to learn in this notebook, but here you can find a class that converts the dataset from the organizers into COCO format with just one call. This way you can just copy the class and use it without going into details. I hope this will be helpful<br>
</div>

![](https://thumbs.gfycat.com/JampackedBothJumpingbean-size_restricted.gif)

<div class="alert alert-block alert-info" style="font-size:20px; background-color: #f4ffab; font-family:verdana; color: #8a0f0f; border: 3px #cbd600 solid">
    <b>Structure of the COCO Dataset</b>
    <br>In the image below you can see the structure of the dataset that you will get after using the class, as well as a small legend for the schema 👇<br>
</div>

![](https://i.ibb.co/jzPRck5/Screenshot-from-2023-06-25-06-02-15.png)

<div class="alert alert-block alert-info" style="font-size:20px; background-color: #f4ffab; font-family:verdana; color: #8a0f0f; border: 3px #cbd600 solid">
    <b>Libraries ⚡️</b>
</div>

In [1]:
from itertools import chain
import json
import os
import shutil
from tqdm.notebook import tqdm
from colorama import Fore
import yaml
import numpy as np

<div class="alert alert-block alert-info" style="font-size:20px; background-color: #f4ffab; font-family:verdana; color: #8a0f0f; border: 3px #cbd600 solid">
    <b>Class for easy converting to COCO format 🤌🏻</b>
</div>

In [2]:
class COCODataset:
    def __init__(self, images_dirpath: str, annotations_filepath: str, length: int = 1633):
        self.train_size = None
        self.val_size = None
        self.length = length
        self.classes = None
        self.labels_counter = None
        self.normalize = None
        
        self.images_dirpath = images_dirpath
        self.annotations_filepath = annotations_filepath
        self.dataset_dirpath = os.path.join(os.getcwd(), "dataset")
        self.train_dirpath =  os.path.join(self.dataset_dirpath, "train")
        self.val_dirpath =  os.path.join(self.dataset_dirpath, "val")
        self.config_path = os.path.join(self.dataset_dirpath, "coco.yaml")

        self.samples = self.parse_jsonl(annotations_filepath)
        self.classes_dict = {
            "blood_vessel": 0,
            "glomerulus": 1,
            "unsure": 2,
        }

    def __prepare_dirs(self) -> None:
        if not os.path.exists(self.dataset_dirpath):
            os.makedirs(os.path.join(self.train_dirpath, "images"), exist_ok=True)
            os.makedirs(os.path.join(self.train_dirpath, "labels"), exist_ok=True)
            os.makedirs(os.path.join(self.val_dirpath, "images"), exist_ok=True)
            os.makedirs(os.path.join(self.val_dirpath, "labels"), exist_ok=True)
        else:
            raise RuntimeError("Dataset already exists!")

    def __define_splitratio(self) -> None:
        self.train_size = round(self.length * self.train_size)
        self.val_size = self.length - self.train_size
        assert self.train_size + self.val_size == self.length

    def parse_jsonl(self, path: str) -> list[dict, ...]:
        with open(path, 'r') as json_file:
            jsonl_samples = [
                json.loads(line)
                for line in tqdm(
                    json_file, desc="Processing polygons", total=self.length
                )
            ]
        return jsonl_samples

    def __define_paths(self, i: int) -> dict:
        data_path = self.val_dirpath
        if i < self.train_size:
            data_path = self.train_dirpath
        return {
            "images": os.path.join(data_path, "images"),
            "labels": os.path.join(data_path, "labels")
        }

    @staticmethod
    def __get_label_path(paths_dict: dict, identifier: str) -> str:
        return os.path.join(
            paths_dict["labels"],
            f"{identifier}.txt"
        )

    @staticmethod
    def __get_image_path(paths_dict: dict, identifier: str) -> str:
        return os.path.join(
            paths_dict["images"],
            f"{identifier}.tif"
        )

    def __copy_image(self, dst_path: str, identifier: str) -> str:
        shutil.copyfile(
            os.path.join(self.images_dirpath, f"{identifier}.tif"),
            dst_path
        )

    def __copy_label(self, annotations: list, dst_path: str) -> None:
        with open(dst_path, "w") as file:
            for annotation in annotations:
                coordinates = annotation["coordinates"][0]
                label = self.classes_dict[annotation["type"]]
                if label in self.classes:
                    if coordinates:
                        if self.normalize:
                            coordinates = np.array(coordinates) / 512.0
                        coordinates = " ".join(map(str, chain(*coordinates)))
                        file.write(f"{label} {coordinates}\n")
                        self.labels_counter += 1

    def __splitfolders(self):
        for i, line in tqdm(
                enumerate(self.samples),
                desc="Dataset creation", total=self.length
        ):
            self.labels_counter = 0
            identifier = line["id"]
            annotations = line["annotations"]
            paths_dict = self.__define_paths(i)

            dst_image_path = self.__get_image_path(paths_dict, identifier)
            dst_label_path = self.__get_label_path(paths_dict, identifier)

            self.__copy_image(dst_image_path, identifier)
            self.__copy_label(annotations, dst_label_path)

            if self.labels_counter == 0:
                os.remove(dst_image_path)
                os.remove(dst_label_path)

    def __count_dataset(self) -> dict:
        train_images = len(os.listdir(os.path.join(self.train_dirpath, "images")))
        train_labels = len(os.listdir(os.path.join(self.train_dirpath, "labels")))
        val_images = len(os.listdir(os.path.join(self.val_dirpath, "images")))
        val_labels = len(os.listdir(os.path.join(self.val_dirpath, "labels")))
        return {
            "train_images": train_images,
            "train_labels": train_labels,
            "val_images": val_images,
            "val_labels": val_labels
        }

    @staticmethod
    def __check_sanity(count_dict: dict) -> None:
        assert count_dict["train_images"] == count_dict["train_labels"]
        assert count_dict["val_images"] == count_dict["val_labels"]

    def __finalizing(self, count_dict: dict) -> None:
        assert os.path.exists(self.dataset_dirpath)

        example_structure = [
            "dataset",
            "train", "labels", "images",
            "val", "labels", "images"
        ]

        dir_bone = (
            dirname.split("/")[-1]
            for dirname, _, filenames in os.walk(self.dataset_dirpath)
            if dirname.split("/")[-1] in example_structure
        )

        try:
            print("\n~ HuBMAP Dataset Structure ~\n")
            print(
            f"""
          ├── {next(dir_bone)}
          │   │
          │   ├── {next(dir_bone)}
          │   │   └── {next(dir_bone)}
          │   │   └── {next(dir_bone)}
          │   │
          │   ├── {next(dir_bone)}
          │   │   └── {next(dir_bone)}
          │   │   └── {next(dir_bone)}
            """
            )
        except StopIteration as e:
            print(e)
        else:
            print(Fore.GREEN + "-> Success")
            print(Fore.GREEN + f"Train dataset: {count_dict['train_images']}\nVal dataset: {count_dict['val_images']}")

    def get_config(self) ->dict:
        names = ["blood_vessel", "glomerulus", "unsure"]
        return {
            "train": str(self.train_dirpath),
            "val": str(self.val_dirpath),
            "names": [names[i] for i in self.classes]
        }

    @staticmethod
    def display_config(config: dict) -> None:
        print(Fore.BLACK + "\n~ HuBMAP Config Structure ~\n")
        print(
        f"""
      │   │
      │   ├── train
      │   │   └── {config['train']}/images
      │   │
      │   │
      │   ├── val
      │   │   └── {config['val']}/images
      │   │
      │   │
      │   ├── names
      │   │   └── {' '.join(config['names'])}
        """
        )
        print(Fore.GREEN + "-> Success")
        print(Fore.GREEN + f"Number of classes: {len(config['names'])}"
                           f"\nClasses: {' '.join(config['names'])}" 
              )

    def write_config(self, config: dict) -> None:
        with open(self.config_path, mode="w") as f:
            yaml.safe_dump(stream=f, data=config)

    def __call__(self, train_size: float,
                 classes: list[int, ...],
                 make_config: bool = True,
                 normalize: bool = True
                ) -> None:
        
        self.train_size = train_size
        self.classes = classes
        self.normalize = normalize
        
        self.__define_splitratio()
        self.__prepare_dirs()
        self.__splitfolders()
        count_dict = self.__count_dataset()
        self.__check_sanity(count_dict)
        self.__finalizing(count_dict)
        
        if make_config:
            config = self.get_config()
            self.write_config(config)
            self.display_config(config)

<div class="alert alert-block alert-info" style="font-size:20px; background-color: #f4ffab; font-family:verdana; color: #8a0f0f; border: 3px #cbd600 solid">
    <b>And this is what the data configuration file looks like. It is very important if you are going to use a framework like ultralytics 👇</b>
</div>

![](https://i.ibb.co/wLrD9Dt/Screenshot-from-2023-06-25-07-35-26.png)

<div class="alert alert-block alert-info" style="font-size:20px; background-color: #f4ffab; font-family:verdana; color: #8a0f0f; border: 3px #cbd600 solid">
    <b>Let's check🏻🚦</b>
</div>

In [3]:
coco = COCODataset(
    annotations_filepath="/kaggle/input/hubmap-hacking-the-human-vasculature/polygons.jsonl",
    images_dirpath="/kaggle/input/hubmap-hacking-the-human-vasculature/train",
) 

Processing polygons:   0%|          | 0/1633 [00:00<?, ?it/s]

In [4]:
coco(train_size=0.80, classes=[0, 1, 2])

Dataset creation:   0%|          | 0/1633 [00:00<?, ?it/s]


~ HuBMAP Dataset Structure ~


          ├── dataset
          │   │
          │   ├── train
          │   │   └── images
          │   │   └── labels
          │   │
          │   ├── val
          │   │   └── images
          │   │   └── labels
            
-> Success
Train dataset: 1306
Val dataset: 327

~ HuBMAP Config Structure ~


      │   │
      │   ├── train
      │   │   └── /kaggle/working/dataset/train/images
      │   │
      │   │
      │   ├── val
      │   │   └── /kaggle/working/dataset/val/images
      │   │
      │   │
      │   ├── names
      │   │   └── blood_vessel glomerulus unsure
        
-> Success
Number of classes: 3
Classes: blood_vessel glomerulus unsure


<div class="alert alert-block alert-info" style="font-size:20px; background-color: #f4ffab; font-family:verdana; color: #8a0f0f; border: 3px #cbd600 solid">
    <b>I hope this was helpful 😀 Should I make a notepad using this dataset in practice? Good luck!</b>
</div>